## 第1週4日目、ラボ3へようこそ

今日は、すぐに価値を実感できるものを作っていきます！これは2日間にわたるラボの始まりです。

そして、Agent Frameworkを使わずに、Agent Loopを手作りしていきます……

### まず、いくつかの準備

`twin`フォルダに、`linkedin.pdf`という1つのファイルを置いています。これは私のLinkedInプロフィールをPDFでダウンロードしたものです。

これを、あなた自身のものに置き換えてください！LinkedInのプロフィールページから、自分の名前の下にあるメニューを使ってダウンロードできるはずです。この機能にアクセスできない場合は、履歴書などの任意のPDFでも構いません。

また、`twin`フォルダには`summary.txt`というファイルも用意しています。これを読んで、あなた自身の内容に更新してください。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">パッケージを調べる</h2>
            <span style="color:#00bfff;">このラボでは、素早くUIを作れる素晴らしいパッケージであるGradioを使用します。
            また、広く使われているPDFリーダーであるPyPDFも使用します。自分のプロジェクトのためにパッケージをどのように選ぶのか気になる場合は、<a href="https://edwarddonner.com/avatar?q=37">FAQ</a>ページのQ37を参照してください。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# これらのパッケージが何をするのか分からない場合は、いつでもChatGPTに解説してもらいましょう！

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## 補足：おさらいとして3つの概念

1. システムプロンプト：LLMへの入力のうち、会話全体の文脈を説明する部分

2. 会話履歴：これまでの会話全体

3. 記憶の幻想：LLMへのメッセージはすべてステートレスです。これまでの会話全体を毎回渡すことで、まるで30秒前に話したことを覚えているかのような幻想を作り出しています……

__詳しくは、私の姉妹コースであるAI Engineer Core Track（第1週）をご覧ください__

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"},
    {"role": "assistant", "content": "Well hi there, Ed. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

## 本筋に戻りましょう！

`linkedin`変数にLinkedInプロフィールが入っています。

`summary`変数に要約が入っています。

では、システムプロンプトを組み立てていきましょう……

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

## OpenAI以外のモデルを使っている方への注意

OpenAI以外のモデルを使用している場合、chat()の先頭にこの行を挿入する必要があるかもしれません：

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# さあ、ここからがツール（TOOLS）です！

まずは1つの関数から始めましょう……

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

## ステップ1 - ツールを記述するJSONを書く


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

## ステップ2 - 新しいchat()関数

ここで、ツール呼び出しを実装します。

実際のところ、少し不格好です。これは、美味しい料理のレシピの材料を見て、その材料が実はごく普通のものだったと気づくようなものです。

ツール呼び出しは「if」文にすぎません。この場合、唯一のツールはemailツールだと仮定して、すべてをハードコーディングしています。

補足：もし「待って、これは自分で覚えておいて自分で実装しないといけないのでは？」と思っているなら、重要なポイントはこうです。これこそがAgent Frameworkが代わりにやってくれる部分なのです。実際には、これを自分で書くことはもう二度とないでしょう。私たちはAgent Frameworkによって、こうしたif文から守られています。それが、Agent Frameworkが「抽象化レイヤー」とよく呼ばれる理由です。

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## ステップ3

Agent Frameworkを使わずに実装する、私たちにとって初めてのAgent Loopです！

変更点：
1. ツール呼び出しが常に1つだけだと仮定するのではなく、forループでツールを1つずつ処理するようにしました
2. `if finish_reason=="tool_calls"`を`while finish_reason=="tool_calls"`に変更しました

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# おめでとうございます！

これで、ツールを備えたAIアシスタントを実装しました。
そして、Agent Frameworkを使わずに、Agent Loopを手作業で組み立てました。
それだけです！

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">1. 複数のLLM呼び出しを追加してみましょう！LLMが返答を作成した後、別のLLM呼び出しを使って、その返答が仕事に関する内容のみに厳密に関連しているかを評価します。<br/><br/>2. これを自分のビジネスに応用してみましょう！自分のビジネス分野についての質問に答えられるAIアシスタントを作り、連絡を取りたい人のメールアドレスを記録するためにツールを使いましょう。
            </span>
        </td>
    </tr>
</table>